In [10]:
# Apply displacement vectors from small MC ROI to large ROI using B-mode tracking
import numpy as np
import nibabel as nib
from scipy.ndimage import shift as scipy_shift, gaussian_filter
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Load your B-mode data (for motion tracking)
# Use the normalized version instead of RAW if there are loading issues
bmode_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p14/wk12/combined_nifti/p14_wk12_Fund_normalized_GUI.nii'
print(f"Loading B-mode from: {bmode_path}")

try:
    bmode_data = nib.load(bmode_path).get_fdata()
except Exception as e:
    print(f"Error loading with nibabel: {e}")
    print("Trying manual load...")
    # Manual NIfTI load as fallback
    import struct
    with open(bmode_path, 'rb') as f:
        header = f.read(352)
        dim = struct.unpack('<8h', header[40:56])
        datatype = struct.unpack('<h', header[70:72])[0]
        
        # Read the rest of the file
        data_bytes = f.read()
        
        # Parse based on datatype (64 = float64, 16 = float32, 4 = int16, 2 = uint8)
        if datatype == 64:
            dtype = np.float64
        elif datatype == 16:
            dtype = np.float32
        elif datatype == 4:
            dtype = np.int16
        elif datatype == 2:
            dtype = np.uint8
        else:
            raise ValueError(f"Unknown datatype: {datatype}")
        
        # Reshape
        shape = [d for d in dim[1:dim[0]+1] if d > 1]
        bmode_data = np.frombuffer(data_bytes, dtype=dtype).reshape(shape)

print(f"B-mode data shape: {bmode_data.shape}")
print(f"B-mode data dtype: {bmode_data.dtype}")

# Load your small motion-corrected ROI (defines the tracking region)
small_mc_roi_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p14/wk12/p14_wk12_mc_new.nii.gz'  # UPDATE THIS PATH
small_mc_roi_seg = nib.load(small_mc_roi_path).get_fdata()

# Load your large (full lesion) ROI  
large_roi_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p14/wk12/p14_wk12_test.nii.gz'  # UPDATE THIS PATH
large_roi_seg = nib.load(large_roi_path).get_fdata()

print(f"Small MC ROI shape (raw): {small_mc_roi_seg.shape}")
print(f"Large ROI shape (raw): {large_roi_seg.shape}")
    

Loading B-mode from: /Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p14/wk12/combined_nifti/p14_wk12_Fund_normalized_GUI.nii
B-mode data shape: (604, 524, 920)
B-mode data dtype: float64
Small MC ROI shape (raw): (1048, 604, 920)
Large ROI shape (raw): (1048, 604, 920)


In [11]:
# Process ROIs if they're from combined side-by-side visualization
def process_combined_roi(roi_mask, target_shape):
    """Split side-by-side mask and transpose to match target shape."""
    if roi_mask.shape[0] == 1048:  # Combined side-by-side
        print(f"  Detected combined side-by-side ROI, splitting...")
        mid = 524
        left_half = roi_mask[:mid, :, :]
        right_half = roi_mask[mid:, :, :]
        
        # Choose half with more ROI data
        if np.sum(left_half > 0) > np.sum(right_half > 0):
            roi_mask = left_half
        else:
            roi_mask = right_half
        
        # Transpose to match image orientation
        if roi_mask.shape != target_shape:
            roi_mask = roi_mask.transpose(1, 0, 2)
    
    return roi_mask

print(f"Original B-mode shape: {bmode_data.shape}")

# Process ROIs to match original bmode shape
target_shape = (bmode_data.shape[0], bmode_data.shape[1], bmode_data.shape[2])
small_mc_roi_seg = process_combined_roi(small_mc_roi_seg, target_shape)
large_roi_seg = process_combined_roi(large_roi_seg, target_shape)

print(f"\nFinal shapes (all in Y, X, T format):")
print(f"  B-mode data: {bmode_data.shape}")
print(f"  Small MC ROI: {small_mc_roi_seg.shape}")
print(f"  Large ROI: {large_roi_seg.shape}")

n_frames = bmode_data.shape[2]
print(f"  Number of frames: {n_frames}")

Original B-mode shape: (604, 524, 920)
  Detected combined side-by-side ROI, splitting...
  Detected combined side-by-side ROI, splitting...

Final shapes (all in Y, X, T format):
  B-mode data: (604, 524, 920)
  Small MC ROI: (604, 524, 920)
  Large ROI: (604, 524, 920)
  Number of frames: 920


In [ ]:
# ============================================================================
# TOP-LEFT CORNER TRACKING FOR TRANSLATIONAL MOTION
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import shift as scipy_shift
import ipywidgets as widgets
from IPython.display import display, clear_output

print("="*70)
print("TRACKING TOP-LEFT CORNER OF MASK")
print("="*70)

# Reference frame
reference_frame = 199
n_frames = bmode_data.shape[2]

# Arrays to store results
top_left_positions = np.zeros((n_frames, 2))
translation_vectors = np.zeros((n_frames, 3))  # (dx, dy, time)

# Track top-left corner
print("\nTracking top-left corner...")
for t in range(n_frames):
    if t % 100 == 0:
        print(f"  Frame {t}/{n_frames}...")
    
    mask_t = small_mc_roi_seg[:, :, t] > 0
    
    if np.sum(mask_t) == 0:
        if t > 0:
            top_left_positions[t] = top_left_positions[t-1]
        continue
    
    y_indices, x_indices = np.where(mask_t)
    top_left_positions[t] = [y_indices.min(), x_indices.min()]

# Calculate translation vectors
ref_top_left = top_left_positions[reference_frame]
print(f"\nReference top-left at: (y={ref_top_left[0]:.1f}, x={ref_top_left[1]:.1f})")

for t in range(n_frames):
    dy = top_left_positions[t, 0] - ref_top_left[0]
    dx = top_left_positions[t, 1] - ref_top_left[1]
    translation_vectors[t] = [dx, dy, t]

print(f"\nTranslation range:")
print(f"  X: [{translation_vectors[:, 0].min():.1f}, {translation_vectors[:, 0].max():.1f}] px")
print(f"  Y: [{translation_vectors[:, 1].min():.1f}, {translation_vectors[:, 1].max():.1f}] px")
print(f"\n✓ Created 'translation_vectors' array: shape {translation_vectors.shape}")

# ============================================================================
# APPLY TRANSLATIONS TO LARGE ROI
# ============================================================================
print("\n" + "="*70)
print("APPLYING TRANSLATIONS TO LARGE ROI")
print("="*70)

large_roi_mc_topleft = np.zeros_like(large_roi_seg)

for t in range(n_frames):
    if t % 100 == 0:
        print(f"  Processing frame {t}/{n_frames}...")
    
    shift_vector = [translation_vectors[t, 1], translation_vectors[t, 0]]
    large_roi_mc_topleft[:, :, t] = scipy_shift(large_roi_seg[:, :, t], shift_vector, order=0, mode='constant', cval=0)

print(f"\n✓ Motion compensation complete!")

# Save results
output_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p14/wk12/p14_wk12_large_roi_mc_topleft.nii.gz'
nib.save(nib.Nifti1Image(large_roi_mc_topleft.astype(np.uint8), np.eye(4)), output_path)
print(f"✓ Saved to: {output_path}")

vectors_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p14/wk12/translation_vectors.npy'
np.save(vectors_path, translation_vectors)
print(f"✓ Translation vectors saved to: {vectors_path}")

# ============================================================================
# INTERACTIVE VISUALIZATION
# ============================================================================
print("\n" + "="*70)
print("INTERACTIVE COMPARISON")
print("="*70)

fig_compare, axes_compare = plt.subplots(1, 3, figsize=(20, 7))
plt.close(fig_compare)

def visualize_comparison(frame_idx):
    for ax in axes_compare:
        ax.clear()
    
    # Panel 1: Small MC ROI with top-left corner
    axes_compare[0].imshow(bmode_data[:, :, frame_idx], cmap='gray')
    axes_compare[0].contour(small_mc_roi_seg[:, :, frame_idx], colors='green', linewidths=2)
    axes_compare[0].plot(top_left_positions[frame_idx, 1], top_left_positions[frame_idx, 0], 'r*', markersize=20)
    axes_compare[0].plot(ref_top_left[1], ref_top_left[0], 'y*', markersize=15, alpha=0.5)
    axes_compare[0].set_title(f'Small MC ROI')
    axes_compare[0].axis('off')
    
    # Panel 2: Large ROI (static)
    axes_compare[1].imshow(bmode_data[:, :, frame_idx], cmap='gray')
    axes_compare[1].contour(large_roi_seg[:, :, frame_idx], colors='red', linewidths=2)
    axes_compare[1].set_title(f'Large ROI (Static)')
    axes_compare[1].axis('off')
    
    # Panel 3: Large ROI (motion corrected)
    axes_compare[2].imshow(bmode_data[:, :, frame_idx], cmap='gray')
    axes_compare[2].contour(large_roi_mc_topleft[:, :, frame_idx], colors='blue', linewidths=2)
    axes_compare[2].contour(small_mc_roi_seg[:, :, frame_idx], colors='green', linewidths=1, linestyles='--', alpha=0.5)
    axes_compare[2].set_title(f'Large ROI (MC)\ndx={translation_vectors[frame_idx, 0]:.1f}, dy={translation_vectors[frame_idx, 1]:.1f}')
    axes_compare[2].axis('off')
    
    fig_compare.suptitle(f'Frame {frame_idx}')
    fig_compare.tight_layout()

frame_slider = widgets.IntSlider(value=reference_frame, min=0, max=n_frames-1, step=1, description='Frame:', continuous_update=False)
output_widget = widgets.Output()

def on_frame_change(change):
    with output_widget:
        clear_output(wait=True)
        visualize_comparison(change['new'])
        display(fig_compare)

frame_slider.observe(on_frame_change, names='value')

display(frame_slider, output_widget)
frame_slider.value = reference_frame


TRACKING TOP-LEFT CORNER OF MASK

Tracking top-left corner...
  Frame 0/920...
  Frame 100/920...
  Frame 200/920...
  Frame 300/920...
  Frame 400/920...
  Frame 500/920...
  Frame 600/920...
  Frame 700/920...
  Frame 800/920...
  Frame 900/920...

Reference top-left at: (y=132.0, x=84.0)

Translation range:
  X: [-48.0, 56.0] px
  Y: [-5.0, 24.0] px

✓ Created 'translation_vectors' array: shape (920, 3)

APPLYING TRANSLATIONS TO LARGE ROI
  Processing frame 0/920...
  Processing frame 100/920...
  Processing frame 200/920...
  Processing frame 300/920...
  Processing frame 400/920...
  Processing frame 500/920...
  Processing frame 600/920...
  Processing frame 700/920...
  Processing frame 800/920...
  Processing frame 900/920...

✓ Motion compensation complete!
✓ Saved to: /Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p14/wk12/p14_wk12_large_roi_mc_topleft.nii.gz
✓ Translation vectors saved to: /Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p14/wk12/translation_ve

IntSlider(value=199, continuous_update=False, description='Frame:', max=919)

Output()